# Otimização de Hiperparâmetros — Optuna

Objetivo: minimizar risco de atraso na decisão de manutenção (k=2 semanas).

Métrica: `P90_k2 + 0.5 × MAE_k2` (validação)

1. Carrega configs e dataset (uma vez)
2. Executa N trials do Optuna
3. Retreina com os melhores hiperparâmetros
4. Avalia no conjunto de teste

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

In [2]:
from api.config.dataset_config import DatasetConfig
from api.config.training_config import TrainingConfig
from api.config.model_config import ModelConfig
from api.config.output_config import OutputConfig
from api.config.optimization_config import OptimizationConfig

from moviasai.data.utils import load_raw_data
from moviasai.forecasting.optimization import HyperparameterOptimizer
from moviasai.forecasting.training_pipeline import (
    TrainingPipeline, MultiHeadTrainingPipeline, MoETrainingPipeline
)

## Configurações

In [3]:
def run_optimization(target: str, pipeline_cls: type[TrainingPipeline])-> tuple[HyperparameterOptimizer, 'optuna.Study', TrainingPipeline]:
    """Executa otimização completa para o target especificado ('km' ou 'h')."""
    dataset_cfg = DatasetConfig.from_yaml('../config/dataset_config.yaml')
    training_cfg = TrainingConfig.from_yaml('../config/training_config.yaml')
    model_cfg = ModelConfig.from_yaml('../config/model_config.yaml')
    output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')
    optuna_cfg = OptimizationConfig.from_yaml('../config/optimization_config.yaml')

    df_daily = load_raw_data(output_cfg.train_data_path(target), target=target)

    optimizer = HyperparameterOptimizer.from_config(
        df_daily=df_daily,
        dataset_cfg=dataset_cfg,
        training_cfg=training_cfg,
        model_cfg=model_cfg,
        output_config=output_cfg,
        optuna_cfg=optuna_cfg,
        target=target,
        pipeline_cls=pipeline_cls,
    )

    # Otimizar
    study = optimizer.optimize()

    # Retreinar com melhores hiperparâmetros (epochs completos)
    pipeline = optimizer.retrain_best(max_epochs=training_cfg.trainer.max_epochs)

    return optimizer, study, pipeline

## Executar otimização

In [8]:
TARGET = 'km'  # 'km' ou 'h'
optimizer, study, pipeline = run_optimization(TARGET, MoETrainingPipeline)

✓ Modelo PKL carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST.pkl
  Nome: stage2_km
  Features: 5
  Classes: 3

✓ Carregado: 6718 veículos, 207084 versões
  - Extractors: 4
    • segmentation_features_km: 5 features
    • weekday_features_km: 49 features
    • month_phase_features_km: 15 features
    • monthly_cycle_features_km: 7 features
  - Classificador: stage2_km
✓ Dataset carregado: C:\Users\f0pi\git\apimovias\data\.dataset_cache\km
  • Amostras: 112,787
  • Features: 78
  • X_recent: (112787, 28)
  • Heads: 4


[I 2026-04-26 01:42:05,996] A new study created in memory with name: optim_km


train_test_split temporal:
  Cutoff:       2025-08-17
  Treino:       98,048 amostras (86.9%) | 2025-03-23 → 2025-08-17 (22 semanas)
  Teste:        14,739 amostras (13.1%) | 2025-08-24 → 2025-09-07 (3 semanas)
train_test_split temporal:
  Cutoff:       2025-07-20
  Treino:       78,675 amostras (80.2%) | 2025-03-23 → 2025-07-20 (18 semanas)
  Teste:        19,373 amostras (19.8%) | 2025-07-27 → 2025-08-17 (4 semanas)
Splits preparados — Treino: 78,675  Val: 19,373  Teste: 14,739

  OTIMIZAÇÃO DE HIPERPARÂMETROS — KM
  30 trials | max_epochs=25
  Objetivo: P90_k2 + 0.5 × MAE_k2 + 0.3 × max(mean_error, 0)
  Restrição: P90_k3 ≤ 1.0 dias
  Split temporal: shuffle=False (forçado)

  TensorBoard: C:\Users\f0pi\git\apimovias\logs\training\km\optuna_tb_moe_km


  0%|          | 0/30 [00:00<?, ?it/s]

Seed set to 13
Seed set to 13
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type       | Params | Mode 
--------------------------------------------------------
0 | encoder_general  | Sequential | 11.9 K | train
1 | encoder_temporal | Sequential | 8.2 K  | train
2 | fusion           | Sequential | 13.9 K | train
3 | head_agg         | Linear     | 388    | train
4 | head_daily       | Linear     | 679    | train
5 | loss_heads_fn    | HuberLoss  | 0      | train
6 | loss_daily_fn    | L1Loss     | 0      | train
7 | delta_heads      | ModuleList | 1.2 K  | train
--------------------------------------------------------
36.2 K    Trainable params
0         Non-trainable params
36.2 K    Total params
0.145     Total estimated model params size (MB)
25        Modules in train mode
0         Modules in eval mode


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


  MÉTRICAS — VAL

  HEADS (agregado semanal):
    head_0 (7d):  MAE=495.05  RMSE=693.89
    head_1 (7d):  MAE=521.31  RMSE=733.63
    head_2 (7d):  MAE=537.44  RMSE=755.26
    head_3 (7d):  MAE=549.17  RMSE=765.58

  DAILY (dias 1–7):
    MAE médio:  141.37
    RMSE médio: 202.56

  DECISÃO DE MANUTENÇÃO:

  Limite: 2 semanas (k=2)
    Erro médio:          +0.78 dias
    Erro absoluto médio: 0.99 dias
    P90 erro:            +3.77 dias
    % atrasos:           21.9%
    % adiantamentos:     3.9%

  Limite: 3 semanas (k=3)
    Erro médio:          +0.04 dias
    Erro absoluto médio: 0.05 dias
    P90 erro:            +0.00 dias
    % atrasos:           1.4%
    % adiantamentos:     0.3%

  Limite: 4 semanas (k=4)
    Erro médio:          -0.00 dias
    Erro absoluto médio: 0.00 dias
    P90 erro:            +0.00 dias
    % atrasos:           0.0%
    % adiantamentos:     0.0%


  MÉTRICAS — TEST

  HEADS (agregado semanal):
    head_0 (7d):  MAE=486.61  RMSE=683.97
    head_1 (7d):  

W0426 02:45:18.843000 52936 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0426 02:45:18.845000 52936 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0426 02:45:18.847000 52936 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `VehicleForecastingSafeMoEModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VehicleForecastingSafeMoEModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Relatório gerado: C:\Users\f0pi\git\apimovias\logs\training\km\training_report_moe_km.pdf
Hiperparâmetros salvos: C:\Users\f0pi\git\apimovias\logs\training\km\best_hparams_moe_km.yaml


## Análise dos resultados

In [5]:
import optuna

# Importância dos hiperparâmetros
importances = optuna.importance.get_param_importances(study)
print('Importância dos hiperparâmetros:')
for param, imp in importances.items():
    print(f'  {param}: {imp:.4f}')

Importância dos hiperparâmetros:
  conv_filters: 0.3605
  hidden_dim: 0.2609
  alpha_daily: 0.1681
  lr: 0.1652
  dropout: 0.0453


In [6]:
# Métricas finais (teste)
metrics = pipeline.metrics
for split_name in ('val', 'test'):
    if split_name not in metrics:
        continue
    maint = metrics[split_name]['maintenance']
    print(f'\n=== {split_name.upper()} ===')
    for k_label in ('k2', 'k3', 'k4'):
        m = maint[k_label]
        k = int(k_label[1])
        print(f"\n  Limite {k} semanas:")
        print(f"    Erro médio:          {m['mean_error']:+.2f} dias")
        print(f"    Erro absoluto médio: {m['mae_days']:.2f} dias")
        print(f"    P90 erro:            {m['p90_error']:+.2f} dias")
        print(f"    % atrasos:           {m['pct_late']:.1f}%")
        print(f"    % adiantamentos:     {m['pct_early']:.1f}%")


=== VAL ===

  Limite 2 semanas:
    Erro médio:          +1.40 dias
    Erro absoluto médio: 1.99 dias
    P90 erro:            +6.91 dias
    % atrasos:           30.3%
    % adiantamentos:     8.2%

  Limite 3 semanas:
    Erro médio:          +0.11 dias
    Erro absoluto médio: 0.19 dias
    P90 erro:            +0.00 dias
    % atrasos:           4.7%
    % adiantamentos:     1.2%

  Limite 4 semanas:
    Erro médio:          -0.02 dias
    Erro absoluto médio: 0.02 dias
    P90 erro:            +0.00 dias
    % atrasos:           0.1%
    % adiantamentos:     0.4%

=== TEST ===

  Limite 2 semanas:
    Erro médio:          +1.43 dias
    Erro absoluto médio: 1.92 dias
    P90 erro:            +6.96 dias
    % atrasos:           29.6%
    % adiantamentos:     6.7%

  Limite 3 semanas:
    Erro médio:          +0.16 dias
    Erro absoluto médio: 0.17 dias
    P90 erro:            +0.00 dias
    % atrasos:           5.0%
    % adiantamentos:     0.2%

  Limite 4 semanas:
    Erro m

In [7]:
# Melhores hiperparâmetros encontrados
print('\nMelhores hiperparâmetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')


Melhores hiperparâmetros:
  alpha_daily: 0.5153319938243195
  lr: 0.000859983363617172
  hidden_dim: 64
  conv_filters: 32
  dropout: 0.0026076909708348806
